# Twitter Airline Sentiment Analysis

**Goal:** Classify tweets about US airlines as Positive, Neutral, or Negative
**Algorithm:** Logistic Regression with TF-IDF features
**Dataset:** [Twitter US Airline Sentiment](https://www.kaggle.com/datasets/crowdflower/twitter-airline-sentiment)

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
%matplotlib inline

## 1. Load Data from Kaggle

In [ ]:
path = kagglehub.dataset_download("crowdflower/twitter-airline-sentiment")
df = pd.read_csv(f"{path}/Tweets.csv")
print ('Shape: %s' % (df.shape,))
print ('Columns: %s' % list(df.columns))

<hr>## 2. Exploratory Data Analysis

In [ ]:
print ('Sentiment distribution:\n%s' % df['airline_sentiment'].value_counts())
print ('\nAirlines:\n%s' % df['airline'].value_counts())
print ('\nMissing values:\n%s' % df.isnull().sum())
print ('\nSample tweets:\n%s' % df['text'].head(3).to_string())

In [ ]:
# Visualize sentiment distribution
plt.figure(figsize=(8, 4))
df['airline_sentiment'].value_counts().plot(kind='bar', color=['red', 'blue', 'green'])
plt.title('Sentiment Distribution')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Sentiment by airline
pd.crosstab(df['airline'], df['airline_sentiment']).plot(kind='bar', figsize=(12, 5))
plt.title('Sentiment by Airline')
plt.tight_layout()
plt.show()

<hr>## 3. Text Preprocessing

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)      # remove URLs
    text = re.sub(r'@\w+', '', text)          # remove @mentions
    text = re.sub(r'[^a-z\s]', '', text)      # keep only letters
    text = re.sub(r'\s+', ' ', text).strip()  # collapse spaces
    return text

df['clean_text'] = df['text'].apply(clean_text)
print ('Before:', df['text'].iloc[0][:80])
print ('After :', df['clean_text'].iloc[0][:80])

<hr>## 4. Feature Extraction (TF-IDF)

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X = vectorizer.fit_transform(df['clean_text']).toarray()

# Encode labels: negative=0, neutral=1, positive=2
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
y = df['airline_sentiment'].map(label_map)

print ('Feature matrix: %s rows, %d features' % (X.shape[0], X.shape[1]))
print ('Top 10 words:', list(vectorizer.get_feature_names_out()[:10]))

<hr>## 5. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print ('Train: %d, Test: %d' % (X_train.shape[0], X_test.shape[0]))

<hr>## 6. Train Model

In [ ]:
model = LogisticRegression(max_iter=1000, multi_class='multinomial')
model.fit(X_train, y_train)
print ('Model trained: %s' % model)

<hr>## 7. Evaluate Performance

In [ ]:
y_pred = model.predict(X_test)

print ('Accuracy: %.4f' % accuracy_score(y_test, y_pred))
print ('\nClassification Report:')
print (classification_report(y_test, y_pred, target_names=['Negative', 'Neutral', 'Positive']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Neutral', 'Positive'],
            yticklabels=['Negative', 'Neutral', 'Positive'])
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

<hr>## 8. Test with Real Tweets

In [ ]:
test_tweets = [
    'Absolutely loved the flight! Amazing service and friendly crew.',
    'Flight delayed 5 hours, worst experience ever, never flying again.',
    'The flight was okay, nothing special but on time.'
]
sentiment_names = ['Negative', 'Neutral', 'Positive']

print ('Custom tweet predictions:')
for tweet in test_tweets:
    cleaned = clean_text(tweet)
    vec = vectorizer.transform([cleaned]).toarray()
    pred = model.predict(vec)[0]
    prob = model.predict_proba(vec)[0]
    print ("  '%s'" % tweet[:50])
    print ("  -> %s (%.1f%%)" % (sentiment_names[pred], prob[pred]*100))
    print ()